### PACOTES

In [54]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import time
import itertools
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
from sklearn.model_selection import StratifiedKFold
from scipy.stats import ks_2samp

import pandas as pd
import itertools
import numpy as np





In [55]:
inicio = time.time()

df_resumo = pd.read_csv('resumo_features.csv')
df = pd.read_csv('creditcard.csv')

df_resumo['Comparacao_dis_whith'] = pd.to_numeric(
    df_resumo['Comparacao_dis_whith'],
    errors='coerce'
)

# Quanto menor p-value -> melhor
df_resumo['MannWhitney_Score'] = -np.log10(
    df_resumo['Comparacao_dis_whith'] + 1e-300
)

# Correlações absolutas
df_resumo['Corr_pearson_abs'] = abs(
    df_resumo['Corr_pearson']
)

df_resumo['Corr_spearman_abs'] = abs(
    df_resumo['Correlacao_spearman']
)

# MÉTRICAS
metricas = [

    'Curva ROC',
    'KS',
    'inf_mutua',
    'divergencia_kl',
    'MannWhitney_Score',
    'Sep_mediana_norm',
    'Var_Prop_Pca',
    'Corr_pearson_abs',
    'Corr_spearman_abs',
    'Entropia',
    'Qnd_Outliers',
    'Normalidade'
]

df_resumo[metricas] = df_resumo[metricas].fillna(0)

# NORMALIZAÇÃO
scaler = MinMaxScaler()

df_norm = df_resumo.copy()

df_norm[metricas] = scaler.fit_transform(
    df_norm[metricas]
)

# SCORE FINAL
df_norm['Score_Final'] = (

    # PRINCIPAIS
    df_norm['Curva ROC'] * 0.25 +
    df_norm['KS'] * 0.20 +
    df_norm['divergencia_kl'] * 0.18 +
    df_norm['inf_mutua'] * 0.15 +

    # IMPORTANTES
    df_norm['MannWhitney_Score'] * 0.10 +
    df_norm['Sep_mediana_norm'] * 0.04 +

    # COMPLEMENTARES
    df_norm['Var_Prop_Pca'] * 0.03 +
    df_norm['Corr_spearman_abs'] * 0.025 +
    df_norm['Corr_pearson_abs'] * 0.015 +

    # BAIXO IMPACTO
    df_norm['Entropia'] * 0.01 +
    df_norm['Qnd_Outliers'] * 0.005 +
    df_norm['Normalidade'] * 0.005
)

# Se Score_Final virar NaN
df_norm['Score_Final'] = df_norm['Score_Final'].fillna(0)


# RANKING

ranking = df_norm.sort_values(
    'Score_Final',
    ascending=False
).reset_index(drop=True)

# posição do ranking
ranking.insert(
    0,
    'Posicao_Rank',
    ranking.index + 1
)


pd.set_option('display.max_columns', None)

display(ranking)

# SALVAR CSV

ranking.to_csv(
    'individual_score_feature_individuais.csv',
    index=False
)

print('\nCSV salvo com sucesso!')

,Posicao_Rank,Feature,Normalidade,Entropia,Var_Prop_Pca,Qnd_Outliers,Sep_mediana_norm,Comparacao_dis_whith,Corr_pearson,Correlacao_spearman,inf_mutua,divergencia_kl,Curva ROC,KS,MannWhitney_Score,Corr_pearson_abs,Corr_spearman_abs,Score_Final
0,1,V14,0.0,0.553865,0.239456,0.361285,1.000000,1.471581e-260,-0.302544,-0.064613,0.984641,1.000000,1.000000,1.000000,1.000000,0.926501,1.000000,0.971122
1,2,V12,0.0,0.585371,0.260180,0.391901,0.796989,8.416027e-247,-0.260593,-0.062870,0.917286,0.663488,0.972118,0.924927,0.946936,0.797689,0.972125,0.863496
2,3,V10,0.0,0.454295,0.308976,0.242474,0.579452,9.611131e-222,-0.216883,-0.059564,0.908347,0.763357,0.919245,0.950893,0.850284,0.663475,0.919255,0.849811
3,4,V4,0.0,0.717355,0.522493,0.284656,0.415041,3.625904e-248,0.133447,0.063045,0.586806,0.603386,0.974920,0.902628,0.952203,0.407282,0.974924,0.787462
4,5,V11,0.0,0.718112,0.271506,0.019917,0.497847,4.910592e-226,0.154876,0.060143,0.820345,0.483988,0.928507,0.889460,0.866838,0.473081,0.928514,0.772521
5,6,V17,0.0,0.479250,0.187965,0.189465,0.870823,9.219384e-124,-0.326481,-0.044335,1.000000,0.786255,0.675706,0.875582,0.472352,1.000000,0.675708,0.760908
6,7,V3,0.0,0.517007,0.599221,0.085872,0.486416,1.211048e-219,-0.192961,-0.059278,0.583784,0.475816,0.914680,0.822726,0.842183,0.590022,0.914681,0.725398
7,8,V16,0.0,0.538713,0.200072,0.208973,0.580474,1.808172e-156,-0.196539,-0.049936,0.733854,0.512276,0.765280,0.800845,0.598510,0.601008,0.765281,0.677428
8,9,V7,0.0,0.198458,0.398858,0.228481,0.346706,1.464234e-146,-0.187257,-0.048308,0.457887,0.463254,0.739240,0.767815,0.560292,0.572508,0.739245,0.602501
9,10,V2,0.0,0.380592,0.710736,0.345377,0.221554,1.650438e-163,0.091289,0.051062,0.366738,0.392804,0.783279,0.732939,0.625663,0.277834,0.783288,0.590156



CSV salvo com sucesso!


### TOP 10


In [56]:

# atribuindo às variáveis
Primeiro_lugar = top10.iloc[0]['Feature']
Segundo_lugar = top10.iloc[1]['Feature']
Terceiro_lugar = top10.iloc[2]['Feature']
Quarto_lugar = top10.iloc[3]['Feature']
Quinto_lugar = top10.iloc[4]['Feature']
Sexto_lugar = top10.iloc[5]['Feature']
Setimo_lugar = top10.iloc[6]['Feature']
Oitavo_lugar = top10.iloc[7]['Feature']
Nono_lugar = top10.iloc[8]['Feature']
Decimo_lugar = top10.iloc[9]['Feature']



for i, row in top10.iterrows():
    print(f"{row['Posicao_Rank']}º -> {row['Feature']}")

1º -> V14
2º -> V12
3º -> V10
4º -> V4
5º -> V11
6º -> V17
7º -> V3
8º -> V16
9º -> V7
10º -> V2


### AVALIANDO COMBINACOES 2X2 DAS FEATURES

In [57]:

# PEGANDO TOP 10 FEATURES

features_top10 = top10['Feature'].tolist()

print("Top 10 Features:\n")

for f in features_top10:
    print(f)

# TARGET
y = df['status_fraude']

# RESULTADOS
resultados = []

# COMBINAÇÕES 2x2

combinacoes = list(
    itertools.combinations(features_top10, 2)
)

print(f'\nTotal de combinações: {len(combinacoes)}\n')

# CROSS VALIDATION

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# LOOP
for f1, f2 in combinacoes:

    try:

        # MATRIZ X
        X = df[[f1, f2]].values

        # LISTAS CV

        roc_scores = []
        ap_scores = []
        ks_scores = []

        # CROSS VALIDATION

        for train_idx, test_idx in cv.split(X, y):

            # SPLIT

            X_train = X[train_idx]
            X_test = X[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            # PADRONIZAÇÃO

            scaler = StandardScaler()

            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            # MODELO

            modelo = LogisticRegression()

            modelo.fit(X_train, y_train)

            # PROBABILIDADES

            probs = modelo.predict_proba(X_test)[:, 1]

            # ROC AUC CV

            roc = roc_auc_score(
                y_test,
                probs
            )

            roc_scores.append(roc)

            # AVERAGE PRECISION SCORE

            ap = average_precision_score(
                y_test,
                probs
            )

            ap_scores.append(ap)

            # KS

            probs_normal = probs[y_test == 0]
            probs_fraude = probs[y_test == 1]

            ks = ks_2samp(
                probs_normal,
                probs_fraude
            ).statistic

            ks_scores.append(ks)

        # MÉDIAS

        roc_final = np.mean(roc_scores)
        ap_final = np.mean(ap_scores)
        ks_final = np.mean(ks_scores)

        # SALVANDO
        resultados.append({

            'Feature_1': f1,
            'Feature_2': f2,

            'CV_ROC_AUC': round(float(roc_final), 6),
            'Average_Precision': round(float(ap_final), 6),
            'KS_Conjunto': round(float(ks_final), 6)

        })

        # PRINT

        print(
            f'{f1} / {f2} '
            f'-> CV_ROC: {roc_final:.6f} '
            f'| AP: {ap_final:.6f} '
            f'| KS: {ks_final:.6f}'
        )

    except Exception as e:

        print(f'Erro em {f1} / {f2}: {e}')

# DATAFRAME FINAL

df_resultados = pd.DataFrame(resultados)

# ORDENANDO

df_resultados = df_resultados.sort_values(
    by='CV_ROC_AUC',
    ascending=False
).reset_index(drop=True)

# RANKING

df_resultados.insert(
    0,
    'Rank',
    df_resultados.index + 1
)


# RESULTADO FINAL

print(df_resultados.head(20))

# SALVAR CSV

df_resultados.to_csv(
    '2x2_score_feature_cv.csv',
    index=False
)

print('\nCSV salvo com sucesso!')

Top 10 Features:

V14
V12
V10
V4
V11
V17
V3
V16
V7
V2

Total de combinações: 45

V14 / V12 -> CV_ROC: 0.961695 | AP: 0.694923 | KS: 0.863809
V14 / V10 -> CV_ROC: 0.949564 | AP: 0.716528 | KS: 0.862572
V14 / V4 -> CV_ROC: 0.972419 | AP: 0.659782 | KS: 0.876625
V14 / V11 -> CV_ROC: 0.952817 | AP: 0.627243 | KS: 0.853084
V14 / V17 -> CV_ROC: 0.949959 | AP: 0.681880 | KS: 0.855108
V14 / V3 -> CV_ROC: 0.954001 | AP: 0.636490 | KS: 0.861005
V14 / V16 -> CV_ROC: 0.951709 | AP: 0.676060 | KS: 0.850208
V14 / V7 -> CV_ROC: 0.949819 | AP: 0.682450 | KS: 0.852877
V14 / V2 -> CV_ROC: 0.952619 | AP: 0.619660 | KS: 0.850634
V12 / V10 -> CV_ROC: 0.940800 | AP: 0.690490 | KS: 0.818385
V12 / V4 -> CV_ROC: 0.963446 | AP: 0.594804 | KS: 0.826466
V12 / V11 -> CV_ROC: 0.941233 | AP: 0.637641 | KS: 0.806694
V12 / V17 -> CV_ROC: 0.912369 | AP: 0.652939 | KS: 0.799606
V12 / V3 -> CV_ROC: 0.944561 | AP: 0.625788 | KS: 0.808502
V12 / V16 -> CV_ROC: 0.919007 | AP: 0.653887 | KS: 0.797461
V12 / V7 -> CV_ROC: 0.933

### COMBINACAO 3X3

In [58]:
y = df['status_fraude']

# TOP 10 FEATURES

features_top10 = top10['Feature'].tolist()

print("Top 10 features:")
print(features_top10)

# COMBINAÇÕES 3x3
combinacoes = list(itertools.combinations(features_top10, 3))

print(f"\nTotal combinações 3x3: {len(combinacoes)}\n")

# CROSS VALIDATION
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resultados = []

# LOOP
for f1, f2, f3 in combinacoes:

    try:

        X = df[[f1, f2, f3]].values

        cv_roc_scores = []
        ap_scores = []
        ks_scores = []

        for train_idx, test_idx in cv.split(X, y):

            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            scaler = StandardScaler()

            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

            modelo = LogisticRegression(max_iter=1000)

            modelo.fit(X_train, y_train)

            probs = modelo.predict_proba(X_test)[:, 1]

            # =========================
            # CV ROC AUC
            # =========================

            cv_roc_scores.append(
                roc_auc_score(y_test, probs)
            )

            # =========================
            # Average Precision
            # =========================

            ap_scores.append(
                average_precision_score(y_test, probs)
            )

            # =========================
            # KS
            # =========================

            ks_scores.append(
                ks_2samp(
                    probs[y_test == 0],
                    probs[y_test == 1]
                ).statistic
            )

        resultados.append({
            'Feature_1': f1,
            'Feature_2': f2,
            'Feature_3': f3,

            'CV_ROC_AUC': np.mean(cv_roc_scores),
            'Average_Precision': np.mean(ap_scores),
            'KS_Conjunto': np.mean(ks_scores)
        })

        print(
            f"{f1}/{f2}/{f3} -> "
            f"CV_ROC: {np.mean(cv_roc_scores):.4f} | "
            f"AP: {np.mean(ap_scores):.4f} | "
            f"KS: {np.mean(ks_scores):.4f}"
        )

    except Exception as e:
        print(f"Erro em {f1}/{f2}/{f3}: {e}")

# DATAFRAME FINAL
df_resultados = pd.DataFrame(resultados)

# ordenar por métrica principal
df_resultados = df_resultados.sort_values(
    by='CV_ROC_AUC',
    ascending=False
).reset_index(drop=True)

df_resultados.insert(0, 'Rank', df_resultados.index + 1)

# SALVAR CSV

df_resultados.to_csv(
    '3x3_score_feature.csv',
    index=False
)

print("\nCSV 3x3 salvo com sucesso!")

Top 10 features:
['V14', 'V12', 'V10', 'V4', 'V11', 'V17', 'V3', 'V16', 'V7', 'V2']

Total combinações 3x3: 120

V14/V12/V10 -> CV_ROC: 0.9536 | AP: 0.7232 | KS: 0.8725
V14/V12/V4 -> CV_ROC: 0.9759 | AP: 0.7071 | KS: 0.8868
V14/V12/V11 -> CV_ROC: 0.9618 | AP: 0.6938 | KS: 0.8659
V14/V12/V17 -> CV_ROC: 0.9604 | AP: 0.7013 | KS: 0.8640
V14/V12/V3 -> CV_ROC: 0.9619 | AP: 0.6941 | KS: 0.8659
V14/V12/V16 -> CV_ROC: 0.9618 | AP: 0.7038 | KS: 0.8594
V14/V12/V7 -> CV_ROC: 0.9585 | AP: 0.7035 | KS: 0.8634
V14/V12/V2 -> CV_ROC: 0.9614 | AP: 0.6939 | KS: 0.8655
V14/V10/V4 -> CV_ROC: 0.9716 | AP: 0.7257 | KS: 0.8812
V14/V10/V11 -> CV_ROC: 0.9503 | AP: 0.7149 | KS: 0.8703
V14/V10/V17 -> CV_ROC: 0.9496 | AP: 0.7264 | KS: 0.8653
V14/V10/V3 -> CV_ROC: 0.9487 | AP: 0.7182 | KS: 0.8621
V14/V10/V16 -> CV_ROC: 0.9504 | AP: 0.7294 | KS: 0.8636
V14/V10/V7 -> CV_ROC: 0.9490 | AP: 0.7181 | KS: 0.8655
V14/V10/V2 -> CV_ROC: 0.9501 | AP: 0.7166 | KS: 0.8619
V14/V4/V11 -> CV_ROC: 0.9725 | AP: 0.6662 | KS: 0.8845


In [59]:
fim = time.time()
tempo_total = fim - inicio

print(f"Tempo de execução: {tempo_total:.4f} segundos")

Tempo de execução: 179.9986 segundos
